<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L02-failure-taxonomy/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L02-failure-taxonomy/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T08-L02-failure-taxonomy/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T08-L02-failure-taxonomy/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T08-L02 · From 100 traces to a failure taxonomy: open coding, executable checks and a CI gate on regressions

**You will build:** a coding sheet and a failure taxonomy from a hundred agent traces, a
measurement of how far two coders agree, the executable checks that turn each failure
category into a binary pass/fail test, and a CI-style gate that decides whether a candidate
change to the agent may ship.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T07-L03-evaluating-agents, T08-L01-agent-observability.

There is no language model and no API here. The traces are generated, with failure modes
planted by a seeded generator that also writes down which ones it planted, so that you can
check your reading of the traces against an answer key. A real application has no key; that
is why the lesson measures how far two readers agree before it trusts either of them.

By the end you will be able to:

1. Implement axial coding: normalise free-text open codes, merge them into categories, and
   count the traces (not the phrases) that fall in each category.
2. Implement Cohen's kappa and use it, with a confusion table, to measure how far two coders
   agree beyond chance, before and after a definition is tightened.
3. Implement three executable binary checks (ungrounded numbers, unsafe refunds, wrong
   entity) and score every check's precision and recall against labelled traces.
4. Implement a CI gate that compares per-check failure rates of a candidate against a
   baseline, and measure what an aggregate pass rate hides and how much run-to-run noise
   the gate has to tolerate.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L03`, `T08-L01`

**This lesson assumes you can already:**

- write Python functions that loop over lists of dictionaries, and use collections.Counter
- read a tool-calling agent trace: the steps, their arguments and results, and the final answer (T07-L01 and T08-L01 build and instrument one)
- say why a failure rate measured on a hundred cases is noisy (T07-L03 builds the bootstrap that measures it)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Frequently Asked Questions (And Answers) About AI Evals](https://hamel.dev/blog/posts/evals-faq/) · Hamel Husain and Shreya Shankar · *free* — the error-analysis loop this lesson follows (open coding, axial coding, binary checks, a repeatable eval set), and the opinions it is built on
- [sklearn.metrics.cohen_kappa_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html) · scikit-learn developers (documentation) · *free* — the definition of Cohen's kappa that exercise 2 implements, with p_o and p_e named
- [decimal: Decimal fixed-point and floating-point arithmetic](https://docs.python.org/3/library/decimal.html) · Python documentation · *free* — why money is compared as a Decimal and not as text or a float, which exercise 3's refund check relies on

In [ ]:
# Setup: everything the lesson needs, in one cell.
import contextlib
import io
import json
import os
import random
import re
import signal
import sys
import threading
import time
import traceback
from collections import Counter
from decimal import Decimal
from typing import Any, Callable, Sequence

print("python", sys.version.split()[0])

DISCOVERY_SEED = "t08l02-discovery-2026-10-02"   # the hundred traces you read
CI_SEED = "t08l02-ci-set-2026-10-02"             # a different hundred tasks, kept for the gate
N_TRACES = 100                                    # both sets have this many traces

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("axial_codes", "taxonomy_table"),
    "exercise 2": ("cohens_kappa",),
    "exercise 3": ("check_ungrounded", "check_unsafe_refund", "check_wrong_entity"),
    "exercise 4": ("check_quality",),
    "exercise 5": ("ci_gate",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (check_ungrounded, ...)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


class CheckTimedOut(BaseException):
    """A check ran past its time limit. A BaseException, so an `except Exception` inside
    the code being checked cannot swallow it and keep looping."""


_CHECK_SECONDS = 2      # an exercise check: small calls that finish in milliseconds
_DEMO_SECONDS = 10      # a demo over generated traces: about a second on a laptop
# Short on purpose: a loop that never ends and appends to a list as it goes grows by hundreds
# of megabytes a second, so the check must stop it within a couple of seconds.


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn code that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in
    the main thread) and does nothing elsewhere. An alarm that was already running (the
    grader's own time limit) keeps its deadline if it is due first and is put back, with the
    time it had left, afterwards. A correct answer finishes far inside the limit, and nothing
    that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s and was stopped: a "
               "loop that never ends? (a loop whose stopping condition is never met)")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise CheckTimedOut(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise CheckTimedOut(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise CheckTimedOut(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except CheckTimedOut as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise CheckTimedOut(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise CheckTimedOut(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except CheckTimedOut as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Why read traces first

A multi-step agent fails in ways nobody listed in advance, so a score you write before you
have read a single trace measures what you already thought of. The error-analysis practice
this lesson follows, as set out in Hamel Husain and Shreya Shankar's evals FAQ, is a loop:
gather traces, **open code** them (write free-text notes on what went wrong, without a
fixed list), **axial code** the notes (group them into a failure taxonomy and count each
group), refine, and only then write evaluators. The FAQ says to prefer a code-based check
when an objective rule can identify the failure, and an LLM judge only when the failure
needs human judgment. It recommends reviewing at least a hundred traces, and binary pass/fail
evaluations rather than 1-5 scales.
Those statements are about a page that is updated, so they are dated: as of 2 October 2026,
the page on which they are read was last modified on 21 September 2026.

T07-L03 counted failures by the agent loop's own terminal status. That is a taxonomy someone
else already wrote. Here the categories come from reading, so they can name what the status
field never could: a refund issued on an order nobody looked up, an answer about the wrong
order. The original eval-skills repository, hamelsmu/evals-skills, was archived by its owner
on 16 August 2026 and its notice says the skills moved to ai-evals-course/evals-skills, the
"evals plugin" that the FAQ now links to; this lesson uses neither. Sources and access dates
are in claims.yaml.

The next cell builds the traces; where your notebook viewer supports it, its source starts
collapsed. You do not need to read it to do the lesson, and reading it before you have coded
the traces spoils the exercise: in real life this cell is your production application.

In [ ]:
CUSTOMERS = [f"c{n:03d}" for n in range(1, 13)]
STATUSES = ["processing", "shipped", "delivered", "cancelled"]
MODES = ["loop", "unrecovered_error", "wrong_entity", "unsafe_refund", "ungrounded"]
# MODES is also the order in which a reader meets a failure: a wrong lookup comes before a
# refund, which comes before the answer. A reader who notes only the FIRST failure in a
# trace, as the FAQ recommends, uses this order.
BASELINE = {"loop": 0.07, "unrecovered_error": 0.10, "wrong_entity": 0.10,
            "unsafe_refund": 0.25, "ungrounded": 0.14, "honest_error": 0.06}
CANDIDATE = {"loop": 0.04, "unrecovered_error": 0.08, "wrong_entity": 0.10,
             "unsafe_refund": 0.45, "ungrounded": 0.06, "honest_error": 0.06}
# Chosen before anything was measured and never tuned: how often each mode fires, per task
# it can happen on. CANDIDATE is "the prompt change under review".
STICKY = 0.7       # chance that a task behaves as it did on every earlier run
TRANSIENT = 0.12   # chance that a healthy tool call times out once and is retried


def _money(cents: int) -> str:
    return f"{cents // 100}.{cents % 100:02d}"


def make_world(seed: str) -> dict:
    """40 orders for 12 customers: id, status, total and ship date, all from the seed."""
    rng = random.Random(f"{seed}/world")
    ids = sorted(rng.sample(range(1000, 10000), 40))
    orders, by_customer = {}, {c: [] for c in CUSTOMERS}
    for n, num in enumerate(ids):
        oid, status = f"ORD-{num}", rng.choice(STATUSES)
        ship = f"2026-{rng.randint(1, 3):02d}-{rng.randint(1, 28):02d}"
        orders[oid] = {"order_id": oid, "status": status,
                       "total": _money(rng.randint(1500, 24999)),
                       "ship_date": ship if status in ("shipped", "delivered") else None}
        by_customer[CUSTOMERS[n % len(CUSTOMERS)]].append(oid)
    return {"orders": orders, "by_customer": by_customer}


def make_task(world: dict, seed: str, i: int) -> dict:
    rng = random.Random(f"{seed}/{i}/task")
    kind = rng.choices(["status", "latest", "refund"], weights=[4, 3, 3])[0]
    ids = sorted(world["orders"])
    if kind == "status":
        oid = rng.choice(ids)
        text = rng.choice([f"Where is order {oid}?", f"What is the status of order {oid}?",
                           f"Can you check order {oid} for me?"])
        return {"kind": kind, "text": text, "order": oid}
    if kind == "latest":
        cid = rng.choice(CUSTOMERS)
        return {"kind": kind, "customer": cid,
                "text": f"I am customer {cid}. What is the status of my latest order?"}
    oid = rng.choice([o for o in ids if world["orders"][o]["status"] != "cancelled"])
    total = world["orders"][oid]["total"]
    cents = int(total.replace(".", ""))
    if rng.random() < 0.5:
        return {"kind": kind, "text": f"Please refund order {oid} in full.", "order": oid,
                "amount": total}
    return {"kind": kind, "text": f"Please refund {_money(cents // 2)} of order {oid}.",
            "order": oid, "amount": _money(cents // 2)}


NUMBER = re.compile(r"\d+(?:[.\-]\d+)*")   # 59.90, 2026-03-04, 4821, 0173: one token each
ORDER_ID = re.compile(r"ORD-\d{4}")
ACKNOWLEDGES = ("could not", "couldn't", "unable", "sorry")


def _draw(seed: str, i: int, mode: str, run: str) -> float:
    """The uniform number that decides whether `mode` fires on task i in this run: shared with
    other runs of the same task STICKY of the time, fresh otherwise."""
    sticky = random.Random(f"{seed}/{i}/{mode}/pick/{run}").random() < STICKY
    key = f"{seed}/{i}/{mode}/sticky" if sticky else f"{seed}/{i}/{mode}/fresh/{run}"
    return random.Random(key).random()


def _agent(world: dict, task: dict, wrong: str | None, unsafe: str | None, pre: list[int]):
    """A scripted agent. It yields each tool call it wants to make and is sent the result;
    what it returns is its final answer."""
    orders = world["orders"]
    if task["kind"] == "status":
        oid = wrong or task["order"]
        res = yield ("get_order", {"order_id": oid})
    elif task["kind"] == "latest":
        found = yield ("search_orders", {"customer": task["customer"]})
        res = yield ("get_order", {"order_id": wrong or found["orders"][-1]})
    else:
        oid = task["order"]
        if unsafe == "blind":
            amount = task["amount"] if task["amount"] != orders[oid]["total"] \
                else _money([2000, 2500, 5000, 10000][pre[0] % 4])
            done = yield ("issue_refund", {"order_id": oid, "amount": amount})
        else:
            info = yield ("get_order", {"order_id": oid})
            amount = task["amount"]
            if unsafe == "over":
                amount = _money(int(info["total"].replace(".", "")) + [500, 1000, 2000][pre[1] % 3])
            done = yield ("issue_refund", {"order_id": oid, "amount": amount})
        return f"Refunded {done['amount']} EUR for order {done['order_id']}. " \
               f"Your refund reference is {done['refund_id']}."
    text = f"Order {res['order_id']} is {res['status']}."
    if res["ship_date"]:
        text += f" It shipped on {res['ship_date']}."
    return text + f" The order total is {res['total']} EUR."


def _tamper(trace_steps: list[dict], task: dict, answer: str, pre: list[int]) -> tuple[str, str]:
    """Make the answer say something no tool result supports. Returns the new answer and how."""
    seen = set(NUMBER.findall(task["text"]))
    for s in trace_steps:
        if s["ok"]:
            seen.update(NUMBER.findall(json.dumps(s["result"])))
    refund = answer.startswith("Refunded")
    dated = re.search(r"\d{4}-\d\d-\d\d", answer)
    options = ["amount" if refund else "total", "promise"]
    if not refund:
        options.append("date" if dated else "invented_date")
    variant = options[pre[6] % len(options)]
    if variant == "promise":
        return answer + " A confirmation email is on its way.", variant
    if variant in ("total", "amount"):
        hit = list(NUMBER.finditer(answer))[0 if variant == "amount" else -1]
        cents, k = int(hit.group(0).replace(".", "")), 1 + pre[2] % 9
        while _money(cents + 100 * k) in seen:
            k += 1
        return answer[:hit.start()] + _money(cents + 100 * k) + answer[hit.end():], variant
    day = pre[3] % 28
    while f"2026-0{1 + day % 3}-{1 + day:02d}" in seen:
        day = (day + 1) % 28
    new = f"2026-0{1 + day % 3}-{1 + day:02d}"
    if variant == "date":
        return answer[:dated.start()] + new + answer[dated.end():], variant
    return answer + f" It shipped on {new}.", variant


def generate_traces(seed: str, n: int, config: dict, run: str = "r0"):
    """n agent traces and the answer key {trace id: [(mode, variant), ...]} of what was planted.

    `config` says how often each failure mode fires. `run` names one execution of the agent:
    the same tasks under a different run name behave mostly, not entirely, the same way."""
    world = make_world(seed)
    traces, planted = [], {}
    for i in range(n):
        task = make_task(world, seed, i)
        rng = random.Random(f"{seed}/{i}/body")
        pre = [rng.randrange(10 ** 6) for _ in range(8)]
        noise = random.Random(f"{seed}/{i}/transient/{run}")
        fires = {m: _draw(seed, i, m, run) < config[m] for m in config}
        kind, truth = task["kind"], []
        stop = "loop" if fires["loop"] else "unrecovered_error" if fires["unrecovered_error"] else None
        wrong = unsafe = None
        tamper = stop is None and fires["ungrounded"]
        if stop is None:
            if fires["wrong_entity"] and kind != "refund":
                pool = [o for o in sorted(world["orders"]) if o != task.get("order")
                        and o not in world["by_customer"].get(task.get("customer"), [])]
                wrong = pool[pre[4] % len(pool)]
                truth.append(("wrong_entity", ""))
            if fires["unsafe_refund"] and kind == "refund":
                unsafe = ("blind", "over")[pre[5] % 2]
                truth.append(("unsafe_refund", unsafe))
        honest = stop is None and not truth and not tamper and fires["honest_error"]
        steps = []

        def emit(tool, args, ok=True, error=None, result=None):
            steps.append({"i": len(steps), "tool": tool, "args": dict(args), "ok": ok,
                          "error": error, "result": result})

        def execute(tool, args):
            if tool == "search_orders":
                return {"customer": args["customer"], "orders": world["by_customer"][args["customer"]]}
            if tool == "get_order":
                return dict(world["orders"][args["order_id"]])
            return {"refund_id": f"R-{1000 + pre[7] % 9000}", "order_id": args["order_id"],
                    "amount": args["amount"]}

        plan_len = 1 if (kind == "status" or (kind == "refund" and unsafe == "blind")) else 2
        fail_at = pre[0] % plan_len
        variant = ("timeout", "invalid_arguments")[pre[1] % 2]
        gen = _agent(world, task, wrong, unsafe, pre)
        call, answer, index = next(gen), None, 0
        while True:
            tool, args = call
            if (stop == "unrecovered_error" or honest) and index == fail_at:
                bad = "timeout" if honest else variant
                shown = args
                if bad == "invalid_arguments":     # the agent forgot an argument
                    shown = {k: v for k, v in args.items() if k == "order_id"} \
                        if tool == "issue_refund" else {}
                emit(tool, shown, False, bad)
                answer = ("I could not reach the order system just now. Please try again later."
                          if honest else "Good news, everything went through.")
                if stop == "unrecovered_error":
                    truth.append(("unrecovered_error", variant))
                break
            if noise.random() < TRANSIENT and stop != "loop":
                emit(tool, args, False, "timeout")
            result = execute(tool, args)
            emit(tool, args, True, None, result)
            if stop == "loop":
                length = (3, 4, 5)[pre[2] % 3]
                for _ in range(length - 1):
                    emit(tool, args, True, None, result)
                truth.append(("loop", f"run{length}"))
                answer = "Sorry, I could not complete this request."
                break
            try:
                call = gen.send(result)
            except StopIteration as done:
                answer = done.value
                break
            index += 1
        if tamper:
            answer, how = _tamper(steps, task, answer, pre)
            truth.append(("ungrounded", how))
        tid = f"T-{i:03d}"
        traces.append({"id": tid, "task": task, "steps": steps, "answer": answer})
        planted[tid] = sorted(truth, key=lambda t: MODES.index(t[0]))
    return traces, planted


discovery, ANSWER_KEY = generate_traces(DISCOVERY_SEED, N_TRACES, BASELINE)
kinds = Counter(t["task"]["kind"] for t in discovery)
print(f"{len(discovery)} traces:", ", ".join(f"{k} {n}" for k, n in sorted(kinds.items())))
print("tools:", ", ".join(sorted({s["tool"] for t in discovery for s in t["steps"]})))
print(f"steps per trace: {min(len(t['steps']) for t in discovery)} to "
      f"{max(len(t['steps']) for t in discovery)}")

## 2. A trace viewer, and ten minutes of reading

The FAQ's advice is to read the raw traces yourself first. A trace here is a dictionary:
the task, a list of steps (tool, arguments, whether the call succeeded, its result or
error) and the agent's final answer. `show_trace` prints one on a few lines. Read a few,
and notice the failed steps that are followed by the same call succeeding: those are
retries, and a reader who flags every error will flag healthy runs.

In [ ]:
def show_trace(trace: dict) -> None:
    print(f"{trace['id']} [{trace['task']['kind']}] {trace['task']['text']}")
    for s in trace["steps"]:
        outcome = ("ok " + json.dumps(s["result"])) if s["ok"] else f"FAILED {s['error']}"
        print(f"  {s['i'] + 1}. {s['tool']} {json.dumps(s['args'])} -> {outcome}")
    print(f"  answer: {trace['answer']}")


for _t in (discovery[0], discovery[1], discovery[2]):
    show_trace(_t)
    print()
_errored = [t for t in discovery if any(not s["ok"] for s in t["steps"])]
print(f"{len(_errored)} of {len(discovery)} traces contain at least one failed tool call")

## 3. Open coding: two readers, two vocabularies

Two scripted coders now read all of the traces and write free-text notes, the way two
people would. They stand in for people: each has fixed habits (what it notices, what it
skims) and a vocabulary of its own, and they use the generator's answer key to decide what
they noticed. Real coders have no key, which is the reason for measuring how far they agree.
Because their habits were written by this lesson's author, every agreement figure below shows
how the measurement works, not how real people agree.
The notes are deliberately untidy: capitals, stray spaces, trailing full stops, two phrasings
for one idea. Run the next cell and read the inventory of phrases before you write any code.

In [ ]:
PHRASES = {
    "loop": ["Keeps calling the same tool with the same arguments", "stuck in a loop."],
    "unrecovered_error": ["tool error ignored, answer carries on",
                          "Error swallowed -  answer sounds fine"],
    "ungrounded_number": ["answer states a number no tool returned", "Made-up figure in the answer"],
    "ungrounded_promise": ["promises an email that no tool sent", "  claims an email nobody sent."],
    "unsafe_blind": ["refund issued without looking the order up", "Refunded blind"],
    "unsafe_over": ["refund larger than the order total", "over-refund."],
    "wrong_entity": ["looked up a different order than the one asked about", "Wrong order fetched"],
    "vague": ["answer looks off", "Answer looks  off."],
}


def _acknowledged(answer: str) -> bool:
    return any(w in answer.lower() for w in ACKNOWLEDGES)


def code_trace(coder: str, trace: dict, definition: int = 1) -> list[str]:
    """The open codes one scripted coder writes for one trace, in the order it met them.

    Coder A reads closely; the one thing it does is treat a run of exactly three identical calls
    as a retry. Coder B skims: it never notices numbers that look like dates, never notices a
    promise, cannot see a refund that is merely too big, and has no word for a wrong lookup
    beyond 'looks off'. B's reading of 'error' depends on its DEFINITION: version 1 is any failed
    call that the answer does not mention; version 2, written after the first comparison, is a
    LAST call that failed and that the answer does not mention."""
    n = int(trace["id"][2:]) % 2
    key = {mode: variant for mode, variant in ANSWER_KEY[trace["id"]]}
    codes = []
    if coder == "A":
        if "loop" in key and key["loop"] != "run3":
            codes.append(PHRASES["loop"][n])
        if "unrecovered_error" in key:
            codes.append(PHRASES["unrecovered_error"][n])
        if "wrong_entity" in key:
            codes.append(PHRASES["wrong_entity"][n])
        if "unsafe_refund" in key:
            codes.append(PHRASES["unsafe_" + key["unsafe_refund"]][n])
        if "ungrounded" in key:
            kind = "promise" if key["ungrounded"] == "promise" else "number"
            codes.append(PHRASES["ungrounded_" + kind][n])
        return codes
    if "loop" in key:
        codes.append(PHRASES["loop"][n])
    failed = [s for s in trace["steps"] if not s["ok"]]
    swallowed = (bool(failed) if definition == 1 else not trace["steps"][-1]["ok"]) \
        and not _acknowledged(trace["answer"])
    if swallowed:
        codes.append(PHRASES["unrecovered_error"][n])
    if "wrong_entity" in key:
        codes.append(PHRASES["vague"][n])
    if key.get("unsafe_refund") == "blind":
        codes.append(PHRASES["unsafe_blind"][n])
    if key.get("ungrounded") in ("total", "amount"):
        codes.append(PHRASES["ungrounded_number"][n])
    return codes


def coding_sheet(coder: str, traces: list[dict], definition: int = 1) -> list[dict]:
    return [{"trace": t["id"], "codes": code_trace(coder, t, definition)} for t in traces]


SHEET_A = coding_sheet("A", discovery)
SHEET_B = coding_sheet("B", discovery)
inventory = Counter(p for sheet in (SHEET_A, SHEET_B) for row in sheet for p in row["codes"])
print(f"{len(inventory)} distinct phrases written; {sum(1 for r in SHEET_A if r['codes'])} of "
      f"{len(SHEET_A)} traces noted by coder A, {sum(1 for r in SHEET_B if r['codes'])} by coder B")
for phrase, count in sorted(inventory.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {count:>3}  {phrase!r}")

## 4. Axial coding

Axial coding decides which phrases mean the same thing and counts each group. The merge table
below is that decision, made by reading the inventory above: both phrasings of an idea go in
one category, with the keys already normalised. One phrase is left out on purpose: "answer
looks off" says nothing about what is wrong, so it earns no category yet.

**Exercise 1.** Write `axial_codes`, which turns one trace's open codes into its categories,
and `taxonomy_table`, which counts how many *traces* each category appears in. The FAQ ends
axial coding with a count per category; it is a count of traces here because the count is
there to show which failure is most common, and a trace described in two phrases is still one
trace.

In [ ]:
MERGE = {
    "keeps calling the same tool with the same arguments": "loop",
    "stuck in a loop": "loop",
    "tool error ignored, answer carries on": "unrecovered_error",
    "error swallowed - answer sounds fine": "unrecovered_error",
    "answer states a number no tool returned": "ungrounded",
    "made-up figure in the answer": "ungrounded",
    "promises an email that no tool sent": "ungrounded",
    "claims an email nobody sent": "ungrounded",
    "refund issued without looking the order up": "unsafe_refund",
    "refunded blind": "unsafe_refund",
    "refund larger than the order total": "unsafe_refund",
    "over-refund": "unsafe_refund",
    "looked up a different order than the one asked about": "wrong_entity",
    "wrong order fetched": "wrong_entity",
}


def primary_label(categories: list[str]) -> str:
    """The label for kappa: the FIRST category a reader met, or "pass" for a clean trace."""
    return categories[0] if categories else "pass"

<details><summary>Hint 1: what to think about</summary>

Two phrases that differ only in capitals, spacing or a final full stop are the same phrase.
Decide the normal form once and apply it before you look anything up.
</details>

<details><summary>Hint 2: the approach in words</summary>

For one trace: normalise each phrase, look it up (a missing phrase gets its own fixed
category name), and keep a category the first time you see it only. For the table: count each
trace's categories once, then order by count, largest first, breaking ties by category name.
</details>

In [ ]:
def axial_codes(codes: Sequence[str], merge: dict[str, str]) -> list[str]:
    """Turn one trace's open codes (free-text phrases) into its axial categories.

    Normalise each phrase first: lowercase it, collapse every run of whitespace to a single
    space, then strip spaces and full stops from both ends. Look the normalised phrase up in
    `merge` (its keys are already normalised); a phrase that is not in `merge` gets the
    category "uncategorised". Return the categories in the order they first appear, each
    once: a trace whose two phrases land in one category has that category once.

    Example:
        >>> axial_codes(["Stuck in a loop.", "  STUCK   in a loop", "weird"],
        ...             {"stuck in a loop": "loop"})
        ['loop', 'uncategorised']

    Returns: a list of category names; [] for a trace with no codes.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def taxonomy_table(sheet: Sequence[dict], merge: dict[str, str]) -> list[tuple[str, int]]:
    """Count, for each axial category, how many TRACES of the coding sheet fall in it.

    `sheet` rows look like {"trace": "T-004", "codes": [phrase, ...]}. Use axial_codes for
    each row. A trace with no codes is a pass and is counted nowhere. Return (category, traces)
    pairs, largest count first, equal counts in the order sorted() gives their names (Python's
    string order, in which every capital letter comes before every lowercase one).

    Example:
        >>> taxonomy_table([{"trace": "a", "codes": ["Loop", "loop."]},
        ...                 {"trace": "b", "codes": ["loop"]}, {"trace": "c", "codes": []}],
        ...                {"loop": "L"})
        [('L', 2)]

    Returns: a list of (str, int) tuples.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_axial_codes() -> None:
    got = axial_codes(["Stuck in a loop.", "  STUCK   in a loop", "weird"], {"stuck in a loop": "loop"})
    assert got == ["loop", "uncategorised"], (
        f"got {got!r}: normalise (lowercase, collapse spaces, strip spaces and full stops), look "
        "up, name an unknown phrase 'uncategorised', and list each category once")
    got = axial_codes(["b-ish", "A-ish", "b-ish.", "a-ish"], {"b-ish": "B", "a-ish": "A"})
    assert got == ["B", "A"], (
        f"got {got!r}: keep categories in the order they first appear, each once (not sorted)")
    assert axial_codes([], {"x": "X"}) == [], "a trace with no codes has no categories"
    got = axial_codes(["tab\tseparated  words\n"], {"tab separated words": "T"})
    assert got == ["T"], f"got {got!r}: whitespace of any kind collapses to one space"


def _check_taxonomy_table() -> None:
    sheet = [{"trace": "a", "codes": ["Alpha", "alpha."]}, {"trace": "b", "codes": ["alpha", "beta"]},
             {"trace": "c", "codes": []}, {"trace": "d", "codes": ["gamma"]},
             {"trace": "e", "codes": ["Beta"]}, {"trace": "f", "codes": ["unknown"]}]
    merge = {"alpha": "A", "beta": "B", "gamma": "G"}
    got = taxonomy_table(sheet, merge)
    assert got == [("A", 2), ("B", 2), ("G", 1), ("uncategorised", 1)], (
        f"got {got!r}: count each trace once per category (a trace with two phrases for A is one "
        "trace), largest count first, ties alphabetical, passes counted nowhere")
    assert taxonomy_table([], merge) == [], "an empty sheet has an empty table"


_try("exercise 1", lambda: (_check_axial_codes(), _check_taxonomy_table()))

The table for each coder, with each category's share of the traces that coder noted at all.
A trace can sit in more than one category, so the shares can add to more than one.

In [ ]:
def _show_pareto(name: str, sheet: list[dict]) -> None:
    table = taxonomy_table(sheet, MERGE)
    failing = sum(1 for row in sheet if row["codes"])
    print(f"coder {name}: {failing} of {len(sheet)} traces noted")
    for category, n in table:
        print(f"  {category:<18} {n:>3} traces  ({n / failing:.0%} of the noted traces)")


_try("taxonomy tables", lambda: (_show_pareto("A", SHEET_A), _show_pareto("B", SHEET_B)),
     needs=("exercise 1",))


def _show_saturation() -> None:
    first: dict[str, int] = {}
    for position, row in enumerate(SHEET_A, start=1):
        for category in axial_codes(row["codes"], MERGE):
            first.setdefault(category, position)
    print("coder A, reading in order: the trace at which each category first appeared")
    for category, position in sorted(first.items(), key=lambda kv: kv[1]):
        print(f"  {category:<18} trace {position}")
    print(f"the last new category appeared at trace {max(first.values())} of {len(SHEET_A)}")


_try("saturation", _show_saturation, needs=("exercise 1",))

Every category has a count, and one has a count that is not a category: "uncategorised" is the
work still to do. Before turning anything into code, ask whether a second person would have
written the same notes. If not, the categories are one reader's opinion.

## 5. How far do two coders agree?

Both coders read the same traces. Is the share of traces on which they gave the same label a
fair measure of agreement? Not if one label is nearly every trace: two readers who both write
"pass" for most traces agree most of the time by accident. **Cohen's kappa** subtracts the
agreement that two coders with their own habits would reach by chance:
`kappa = (p_o - p_e) / (1 - p_e)`, where `p_o` is the share of traces on which they gave the
same label and `p_e` is the agreement expected from how often each coder used each label
(the sum, over labels, of the product of the two coders' shares).
The FAQ's default is a single domain expert; it names this statistic for the case where you
do use several annotators, which is why two scripted ones are used here. The scikit-learn
documentation gives the same formula (claims.yaml has both, with dates).

**Exercise 2.** Write `cohens_kappa(labels_a, labels_b)` for any number of labels. Two edge
cases need a decision, and this lesson makes it: lists of different lengths, or empty lists,
raise `ValueError`; and when both coders used one single, identical label throughout, the
formula divides by zero and the function returns `1.0`.

<details><summary>Hint 1: what to think about</summary>

Two numbers: how often the labels matched, and how often they would match if each coder
picked labels independently with their own frequencies. A label only one coder used
contributes nothing to the second number.
</details>

<details><summary>Hint 2: the approach in words</summary>

Count each coder's labels, multiply the two shares label by label and add them up for the
expected agreement; compare the observed agreement with that, scaled by the most there is
to gain over chance. Handle the one case where that scale is zero.
</details>

In [ ]:
def cohens_kappa(labels_a: Sequence[Any], labels_b: Sequence[Any]) -> float:
    """Cohen's kappa for two coders' labels of the same items.

    kappa = (p_o - p_e) / (1 - p_e): p_o is the share of items with equal labels, p_e is the sum
    over labels of (share of items coder A gave it) * (share coder B gave it).
    Raise ValueError if the lists differ in length or are empty. If p_e is 1 (both coders used
    one identical label for every item) return 1.0.

    Example: a = [x, x, y, y], b = [x, y, y, y]: p_o = 3/4; p_e = (2/4)(1/4) + (2/4)(3/4) = 1/2;
    kappa = (3/4 - 1/2) / (1 - 1/2) = 0.5.

    Returns: a float, 1.0 for perfect agreement, 0.0 for chance level, negative below chance.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cohens_kappa() -> None:
    got = cohens_kappa(["x", "x", "y", "y"], ["x", "y", "y", "y"])
    assert abs(got - 0.5) < 1e-9, f"got {got!r}, expected 0.5: kappa = (p_o - p_e) / (1 - p_e)"
    a = ["pass"] * 8 + ["f", "g"]
    b = ["pass"] * 8 + ["g", "f"]
    got = cohens_kappa(a, b)
    assert abs(got - (0.8 - 0.66) / (1 - 0.66)) < 1e-9, (
        f"got {got!r}: raw agreement is 0.8 here but most items share one label, so p_e is large; "
        "p_e adds, over every label, share_A * share_B")
    assert cohens_kappa(["u", "v", "u"], ["u", "v", "u"]) == 1.0, "identical labels give 1.0"
    assert cohens_kappa(["q"] * 5, ["q"] * 5) == 1.0, (
        "one identical label used by both coders everywhere: p_e is 1, return 1.0 by this lesson's rule")
    got = cohens_kappa(["q"] * 5, ["r"] * 5)
    assert got == 0.0, (
        f"got {got!r}: each coder used one label, but not the SAME one: p_o and p_e are both 0, so "
        "kappa is 0.0; the 1.0 rule is only for p_e = 1")
    try:
        cohens_kappa(["a", "b"], ["a"])
    except ValueError:
        pass
    else:
        raise AssertionError("lists of different lengths must raise ValueError")


_try("exercise 2", _check_cohens_kappa)

In [ ]:
def confusion(labels_a: Sequence[str], labels_b: Sequence[str]) -> list[tuple[str, str, int]]:
    """The disagreeing (A's label, B's label) pairs, most frequent first."""
    pairs = Counter((x, y) for x, y in zip(labels_a, labels_b) if x != y)
    return sorted(((x, y, n) for (x, y), n in pairs.items()), key=lambda t: (-t[2], t[0], t[1]))


def _labels(sheet: list[dict]) -> list[str]:
    return [primary_label(axial_codes(row["codes"], MERGE)) for row in sheet]


def _show_agreement() -> None:
    labels_a, labels_b = _labels(SHEET_A), _labels(SHEET_B)
    raw = sum(x == y for x, y in zip(labels_a, labels_b)) / len(labels_a)
    print(f"definition 1: the coders gave the same label to {raw:.0%} of traces; "
          f"kappa {cohens_kappa(labels_a, labels_b):.3f}")
    print("where they disagree (A's label / B's label: traces):")
    for x, y, n in confusion(labels_a, labels_b):
        print(f"  {x:<18} / {y:<18} {n:>3}")
    sheet_b2 = coding_sheet("B", discovery, definition=2)
    labels_b2 = _labels(sheet_b2)
    raw2 = sum(x == y for x, y in zip(labels_a, labels_b2)) / len(labels_a)
    k1, k2 = cohens_kappa(labels_a, labels_b), cohens_kappa(labels_a, labels_b2)
    print(f"definition 2 (B reads 'error' as a last call that failed and went unmentioned): "
          f"same label {raw2:.0%}; kappa {k2:.3f}, which {'rose' if k2 > k1 else 'fell' if k2 < k1 else 'did not move'} "
          f"from {k1:.3f}")
    print("what is left:")
    for x, y, n in confusion(labels_a, labels_b2):
        print(f"  {x:<18} / {y:<18} {n:>3}")


_try("agreement", _show_agreement, needs=("exercise 1", "exercise 2"))

Compare the two kappa values printed above, and remember what they are: properties of two
scripts, not evidence about people. Coder B's second definition was written to remove one
disagreement, and it does. What the comparison shows is the mechanism: a disagreement about
what a word means, such as "error", is the kind a written definition fixes.
The disagreements that are left are the ones the scripts were given, and the second definition
does not move them: coder B has no word for a wrong lookup beyond "looks off", cannot see a refund that is
merely too big, and skims past a promise or a date, and coder A reads a run of exactly three
identical calls as a retry. Habits like these are hard to write into a definition, and a check
reads every trace the same way: that is the next step. The FAQ's advice on how annotators
should resolve disagreements is the loop you just ran: label independently, measure agreement,
find the rule that caused each disagreement, write it into the definition.

## 6. From categories to executable checks

A category becomes a **binary check**: a function that reads one trace and answers True when
the failure is present. This follows the FAQ: where an objective rule can identify the
failure, write code, and give it passing and failing examples. Two checks are written for you
as worked examples. `check_loop` finds a run of identical consecutive calls (a single retry is
a run of two, so the threshold is three). `check_unrecovered_error` flags a trace whose last
call failed and whose answer says nothing about it.

In [ ]:
def longest_run(trace: dict) -> int:
    """The longest run of consecutive steps with the same tool and the same arguments."""
    longest = run = 0
    previous = None
    for step in trace["steps"]:
        signature = (step["tool"], json.dumps(step["args"], sort_keys=True))
        run = run + 1 if signature == previous else 1
        longest, previous = max(longest, run), signature
    return longest


def check_loop(trace: dict, min_run: int = 3) -> bool:
    """True if some tool is called with identical arguments at least `min_run` times in a row."""
    return longest_run(trace) >= min_run


def check_unrecovered_error(trace: dict) -> bool:
    """True if the last tool call failed and the final answer does not acknowledge it."""
    return not trace["steps"][-1]["ok"] and not _acknowledged(trace["answer"])


print("check_loop on trace T-000:", check_loop(discovery[0]))
_loops = [t["id"] for t in discovery if check_loop(t)]
print(f"check_loop flags {len(_loops)} of {len(discovery)} traces")

**Exercise 3.** Write the other three, one per remaining category. Each takes a trace
(the dictionary `show_trace` prints) and returns a bool, True meaning the failure is present.

* `check_ungrounded(trace)`: the answer states a number that is not in the task text and
  not in the result of a step that succeeded. Numbers are the `NUMBER` tokens above, compared
  as whole tokens: `9.90` is not grounded by `59.90`. The agent's own arguments and the text of
  failed calls are not evidence. (This check reads numbers only; section 7 measures what that
  leaves uncovered.)
* `check_unsafe_refund(trace)`: some *successful* `issue_refund` step has no successful
  `get_order` for the same order id before it, or refunds more than that order's total. A
  refund of exactly the total is fine; a refund call that failed moved no money. Amounts are
  compared as `Decimal`s, and the refund's own result says which order and how much.
* `check_wrong_entity(trace)`: some `get_order` call asks for an order id that the task text
  did not mention and that no earlier successful `search_orders` returned. A call with no
  `order_id` in its arguments is a different failure and is not flagged here.

<details><summary>Hint 1: what to think about</summary>

Each check is a small amount of bookkeeping over the steps in order: what has been verified so
far, which ids have been legitimately seen so far, which numbers count as evidence. Decide what
each of those collections holds before you scan the steps.
</details>

<details><summary>Hint 2: the approach in words</summary>

Walk the steps once, in order, updating the collection whenever a step qualifies, and test
each step that could be the failure against the collection as it stands at that moment. For
the ungrounded check, build the set of evidence tokens first, then test every token of the
answer.
</details>

In [ ]:
def check_ungrounded(trace: dict) -> bool:
    """True if the answer states a number that the task and the successful results do not.

    Evidence is the NUMBER tokens of the task text and of json.dumps(result) for every step
    with ok True. Compare whole tokens, not substrings. Arguments and failed steps are not
    evidence.

    Example: task "Where is order ORD-1111?", one ok step with result
    {"order_id": "ORD-1111", "total": "20.00"}, answer "Order ORD-1111 total 21.00 EUR":
    the token 21.00 has no evidence -> True.

    Returns: a bool.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def check_unsafe_refund(trace: dict) -> bool:
    """True if a successful issue_refund was not preceded by a successful get_order of the same
    order, or refunded more than that order's total.

    Use each result's own "order_id", "total" (get_order) and "amount" (issue_refund); compare
    amounts as Decimal. Refunding exactly the total is allowed. Failed steps do not count in
    either direction.

    Example: get_order result total "30.00", then issue_refund result amount "30.01" for the
    same order -> True; amount "30.00" -> False; no get_order at all -> True.

    Returns: a bool.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def check_wrong_entity(trace: dict) -> bool:
    """True if a get_order call names an order the task and the earlier searches never produced.

    Allowed ids: the ORDER_ID matches in the task text, plus every id in the result's "orders"
    of a successful search_orders step EARLIER in the trace. Every get_order call counts, failed
    or not; one whose arguments have no "order_id" is skipped.

    Example: task "Where is order ORD-1111?", get_order for ORD-1112 -> True; for ORD-1111 -> False.

    Returns: a bool.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _mk(text: str, steps: list[dict], answer: str, kind: str = "status") -> dict:
    return {"id": "T-x", "task": {"kind": kind, "text": text}, "steps": steps, "answer": answer}


def _ok(tool: str, args: dict, result: dict) -> dict:
    return {"i": 0, "tool": tool, "args": args, "ok": True, "error": None, "result": result}


def _bad(tool: str, args: dict, error: str = "timeout") -> dict:
    return {"i": 0, "tool": tool, "args": args, "ok": False, "error": error, "result": None}


def _order(oid: str, total: str = "20.00", ship: str | None = "2026-02-03") -> dict:
    return {"order_id": oid, "status": "shipped", "total": total, "ship_date": ship}


def _check_ungrounded() -> None:
    steps = [_ok("get_order", {"order_id": "ORD-1111"}, _order("ORD-1111"))]
    ask = "Where is order ORD-1111?"
    assert not check_ungrounded(_mk(ask, steps, "Order ORD-1111 total 20.00 EUR, shipped 2026-02-03.")), (
        "a number that is in a successful result is grounded; so is an id the task itself gave")
    assert check_ungrounded(_mk(ask, steps, "Order ORD-1111 total 21.00 EUR.")), (
        "21.00 is in neither the task nor any result: the answer is ungrounded")
    big = [_ok("get_order", {"order_id": "ORD-1111"}, _order("ORD-1111", "120.00"))]
    assert check_ungrounded(_mk(ask, big, "The total is 20.00 EUR.")), (
        "compare whole numbers: 20.00 is not grounded by 120.00 just because it is a substring of it")
    assert check_ungrounded(_mk(ask, [_bad("get_order", {"order_id": "ORD-1111", "x": "55.00"})],
                                "It is 55.00 EUR.")), (
        "the agent's own arguments are not evidence, and a failed call produced no result")
    assert not check_ungrounded(_mk(ask, steps, "Nothing to report.")), "an answer with no numbers is grounded"


def _check_unsafe_refund() -> None:
    ask = "Please refund order ORD-1111 in full."
    look = _ok("get_order", {"order_id": "ORD-1111"}, _order("ORD-1111", "30.00"))
    pay = lambda amount, oid="ORD-1111": _ok(
        "issue_refund", {"order_id": oid, "amount": amount},
        {"refund_id": "R-1", "order_id": oid, "amount": amount})
    assert not check_unsafe_refund(_mk(ask, [look, pay("30.00")], "ok", "refund")), (
        "refunding exactly the order total after looking it up is safe")
    assert check_unsafe_refund(_mk(ask, [look, pay("30.01")], "ok", "refund")), (
        "30.01 is more than the order total of 30.00: unsafe")
    assert check_unsafe_refund(_mk(ask, [pay("10.00")], "ok", "refund")), (
        "a refund with no earlier successful get_order of that order is unsafe, however small")
    assert check_unsafe_refund(_mk(ask, [pay("10.00"), look], "ok", "refund")), (
        "the lookup must come BEFORE the refund")
    assert check_unsafe_refund(_mk(ask, [look, pay("5.00", "ORD-2222")], "ok", "refund")), (
        "looking up one order does not verify a refund of another")
    assert check_unsafe_refund(_mk(ask, [_bad("get_order", {"order_id": "ORD-1111"}), pay("5.00")],
                                   "ok", "refund")), "a get_order that failed verified nothing"
    assert not check_unsafe_refund(_mk(ask, [look, _bad("issue_refund", {"order_id": "ORD-1111"})],
                                       "sorry", "refund")), "a refund call that failed moved no money"


def _check_wrong_entity() -> None:
    ask = "Where is order ORD-1111?"
    assert not check_wrong_entity(_mk(ask, [_ok("get_order", {"order_id": "ORD-1111"}, _order("ORD-1111"))], "ok")), (
        "an id the task mentioned is allowed")
    assert check_wrong_entity(_mk(ask, [_ok("get_order", {"order_id": "ORD-1112"}, _order("ORD-1112"))], "ok")), (
        "ORD-1112 was never mentioned or found: the wrong entity")
    found = _ok("search_orders", {"customer": "c001"}, {"customer": "c001", "orders": ["ORD-3333"]})
    who = "I am customer c001. What is the status of my latest order?"
    assert not check_wrong_entity(_mk(who, [found, _ok("get_order", {"order_id": "ORD-3333"},
                                                       _order("ORD-3333"))], "ok", "latest")), (
        "an id returned by an earlier successful search is allowed")
    assert check_wrong_entity(_mk(who, [_ok("get_order", {"order_id": "ORD-3333"}, _order("ORD-3333")),
                                        found], "ok", "latest")), (
        "the search must come BEFORE the lookup that relies on it")
    assert not check_wrong_entity(_mk(ask, [_bad("get_order", {}, "invalid_arguments")], "sorry")), (
        "a call with no order_id is a different failure, not a wrong entity")


def _check_exercise_3() -> None:
    _check_ungrounded()
    _check_unsafe_refund()
    _check_wrong_entity()


_try("exercise 3", _check_exercise_3)

## 7. Is a check any good? Precision and recall against labelled traces

A check you have not tested against labelled examples is another opinion. Run it on the
traces whose labels you trust (here the answer key; in practice your adjudicated coding
sheet) and count: **true positives** (flagged, and the failure is there), **false positives**
(flagged, but it is not) and **false negatives** (missed). Precision is the share of what it
flags that is real; recall is the share of the real failures it finds.

**Exercise 4.** Write `check_quality(flagged, truth)` for two equal-length lists of bools.
Return a dict with the three counts under `tp`, `fp`, `fn` and the two ratios under
`precision` and `recall`. A ratio whose denominator is zero (nothing flagged, or nothing real
to find) is `None`, not 0.0 and not 1.0: the check has not been tested on that.

<details><summary>Hint 1: what to think about</summary>

Three counts from pairing each flag with its label; both ratios are a count over a sum of two
counts. Which pairs make each denominator?
</details>

<details><summary>Hint 2: the approach in words</summary>

Walk the two lists together. A flag with a real failure is a hit, a flag without one is a
false alarm, no flag with a real failure is a miss. Divide only when the denominator is not
zero.
</details>

In [ ]:
def check_quality(flagged: Sequence[bool], truth: Sequence[bool]) -> dict:
    """Score one check against labels.

    precision = tp / (tp + fp); recall = tp / (tp + fn); each is None when its denominator is 0.
    Raise ValueError if the lists differ in length.

    Example: flagged [T, T, F, F], truth [T, F, T, F] -> tp 1, fp 1, fn 1, precision 0.5, recall 0.5.

    Returns: {"tp": int, "fp": int, "fn": int, "precision": float | None, "recall": float | None}.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_check_quality() -> None:
    got = check_quality([True, True, True, False, False, False], [True, True, False, True, False, False])
    assert (got["tp"], got["fp"], got["fn"]) == (2, 1, 1), f"got counts {got!r}: tp 2, fp 1, fn 1 expected"
    assert abs(got["precision"] - 2 / 3) < 1e-12 and abs(got["recall"] - 2 / 3) < 1e-12, (
        f"got {got!r}: precision is tp over everything flagged, recall is tp over everything real")
    got = check_quality([False, False], [False, False])
    assert got["precision"] is None and got["recall"] is None, (
        f"got {got!r}: with nothing flagged and nothing real there is nothing to divide: None, not 0.0 or 1.0")
    got = check_quality([False, False], [True, False])
    assert got["precision"] is None and got["recall"] == 0.0, (
        f"got {got!r}: nothing flagged leaves precision undefined (None); a real failure missed is recall 0.0")


_try("exercise 4", _check_check_quality)

In [ ]:
def available_checks() -> dict[str, Callable[[dict], bool]]:
    return {"loop": check_loop, "unrecovered_error": check_unrecovered_error,
            "wrong_entity": check_wrong_entity, "unsafe_refund": check_unsafe_refund,
            "ungrounded": check_ungrounded}


def applies(name: str, trace: dict) -> bool:
    """unsafe_refund can only happen on a refund task, wrong_entity only on the others."""
    kind = trace["task"]["kind"]
    return kind == "refund" if name == "unsafe_refund" else kind != "refund" if name == "wrong_entity" else True


def run_checks(traces: list[dict], checks: dict | None = None) -> dict[str, tuple[int, int]]:
    """{check name: (traces it flagged, traces it applied to)}."""
    out = {}
    for name, fn in (checks or available_checks()).items():
        pool = [t for t in traces if applies(name, t)]
        out[name] = (sum(1 for t in pool if fn(t)), len(pool))
    return out


def naive_repeat(trace: dict) -> bool:
    """A tempting loop check: any tool called more than once."""
    tools = [s["tool"] for s in trace["steps"]]
    return len(tools) != len(set(tools))


def naive_any_error(trace: dict) -> bool:
    """A tempting error check: any failed call at all."""
    return any(not s["ok"] for s in trace["steps"])


def _show_check_quality() -> None:
    def row(label, name, fn):
        pool = [t for t in discovery if applies(name, t)]
        flagged = [bool(fn(t)) for t in pool]
        truth = [any(m == name for m, _ in ANSWER_KEY[t["id"]]) for t in pool]
        q = check_quality(flagged, truth)
        show = lambda v: "n/a" if v is None else f"{v:.2f}"
        print(f"  {label:<26} flagged {sum(flagged):>3}  real {sum(truth):>3}  "
              f"tp {q['tp']:>3} fp {q['fp']:>3} fn {q['fn']:>3}  "
              f"precision {show(q['precision'])}  recall {show(q['recall'])}")
    print("each check against the answer key, on the traces it applies to:")
    for name, fn in available_checks().items():
        row(name, name, fn)
    row("naive: any repeated tool", "loop", naive_repeat)
    row("naive: any failed call", "unrecovered_error", naive_any_error)


_try("check quality", _show_check_quality,
     needs=("exercise 3", "exercise 4"))

The two naive rows are the checks it is tempting to write first. Their recall stays high, and their
precision falls because they also flag every healthy trace that retried a call: a gate built on
a check that cries wolf gets switched off. Most of the other rows score so well because the answer key
here is generated from the same definitions the checks implement: the generator plants a long
run of identical calls and the check looks for one. So perfect scores here show that your code
implements the definition, not that the definition finds real failures. Real traces rarely
agree with a rule so neatly; on them the labels come from your adjudicated coding sheet, and
those are the numbers to look at. The naive rows still show the mechanism worth keeping: a
loose rule cries wolf on healthy retries, and only a labelled set tells you so.

Look at `ungrounded`. It misses failures. The planted failures in that category include
answers that promise an email no tool sent, which has no number in it. This is the honest limit
of a code check: it finds what the rule can see. For the rest you either write a better rule,
or, only where the failure needs judgment, build an LLM judge and validate it against human
labels the same way. The next cell lists what the number check cannot see.

In [ ]:
def _show_ungrounded_misses() -> None:
    missed = [(t["id"], dict(ANSWER_KEY[t["id"]])["ungrounded"]) for t in discovery
              if "ungrounded" in dict(ANSWER_KEY[t["id"]]) and not check_ungrounded(t)]
    kinds = Counter(v for _, v in missed)
    print(f"check_ungrounded misses {len(missed)} planted failures; by the way they were planted:",
          dict(sorted(kinds.items())))
    if missed:
        show_trace(next(t for t in discovery if t["id"] == missed[0][0]))
    loops_a = {r["trace"] for r in SHEET_A if "loop" in axial_codes(r["codes"], MERGE)}
    differ = [t for t in discovery if check_loop(t) and t["id"] not in loops_a]
    lengths = sorted({longest_run(t) for t in differ})
    print(f"check_loop flags {len(differ)} traces that coder A did not note as a loop; the longest run "
          f"of identical calls in them is {lengths}: coder A read a run that short as a retry, and "
          "the check had to decide")


_try("ungrounded misses", _show_ungrounded_misses, needs=("exercise 1", "exercise 3"))

## 8. A candidate change, and the CI set

The agent's owners want to ship a prompt change that saves tokens: fewer verification calls
and a shorter answer. The change is `CANDIDATE` above: it lowers the rate of some failure
modes and raises one, as you can read from the two dictionaries. To decide
whether to ship it you need a **CI set**: tasks you keep, run both versions of the agent on, and
score with the same checks every time. The next cell runs both versions on a fresh set of tasks
(not the ones you read) and prints each check's failure rate. Each check is rated over the
traces it applies to, so the denominators differ.

In [ ]:
BASE_RUN, CAND_RUN = "base", "cand"
ci_base, _ = generate_traces(CI_SEED, N_TRACES, BASELINE, run=BASE_RUN)
ci_cand, _ = generate_traces(CI_SEED, N_TRACES, CANDIDATE, run=CAND_RUN)


def failing_share(traces: list[dict]) -> float:
    """The share of traces that at least one applicable check flags."""
    checks = available_checks()
    bad = sum(1 for t in traces if any(applies(n, t) and fn(t) for n, fn in checks.items()))
    return bad / len(traces)


def _show_ci_reports() -> None:
    base, cand = run_checks(ci_base), run_checks(ci_cand)
    print(f"{'check':<20}{'baseline':>14}{'candidate':>14}")
    for name in base:
        (bf, bn), (cf, cn) = base[name], cand[name]
        print(f"{name:<20}{f'{bf}/{bn}  {bf / bn:.0%}':>14}{f'{cf}/{cn}  {cf / cn:.0%}':>14}")


_try("ci reports", _show_ci_reports, needs=("exercise 3",))

## 9. The gate

A CI gate turns two such reports into one decision and an exit code. The rule here is one
check at a time: a check **regresses** when its candidate failure rate exceeds its baseline
rate by *more than* a margin (strictly more; the margin is in rate, so `0.05` is five
percentage points). The gate fails if any check regresses. The margin used in the demo is
`MARGIN`, a round number picked before looking at any data; section 10 puts it to the test.

**Exercise 5.** Write `ci_gate(baseline, candidate, margin)`. Both reports map a check name to
`(failures, traces)`, as `run_checks` returns. Compare *rates*, because the two runs may have
scored different numbers of traces. Take each check's rise as
`round(candidate_rate - baseline_rate, 9)`: floating-point subtraction can put a rise that equals
the margin on paper a hair above it (the cell after the hints prints one), and the rounding
makes rises that are equal on paper equal in your code. Use that rounded rise both for
the comparison with the margin and for the biggest-first order. Two cases are decided for you. A check present in the
baseline but absent from the candidate report, or scored on zero traces there, is **missing**,
and a missing check fails the gate: deleting the check that was failing is the cheapest
regression there is. A check that was scored on zero baseline traces, or that is new in the
candidate, has no baseline to protect and is ignored.

<details><summary>Hint 1: what to think about</summary>

Which direction counts as bad, and whether a difference exactly equal to the margin is a
regression. Also what each report holds for a check: a count and a denominator, not a rate.
</details>

<details><summary>Hint 2: the approach in words</summary>

Loop over the baseline's checks in name order. Skip the ones with no baseline traces; record the
ones the candidate lost; for the rest turn both pairs into rates and compare the rounded
difference with the margin. Sort the regressions biggest rise first, ties by name.
</details>

In [ ]:
# Why the exercise says to round: the same subtraction, as this machine does it.
_rise = 0.20 - 0.15
print(f"0.20 - 0.15 is {_rise!r} in floating point, which is "
      f"{'more' if _rise > 0.05 else 'not more'} than 0.05, though the two agree on paper")

In [ ]:
def ci_gate(baseline: dict[str, tuple[int, int]], candidate: dict[str, tuple[int, int]],
            margin: float) -> dict:
    """Decide whether a candidate's per-check failure rates are acceptable against a baseline.

    A check regresses when its rise, round(candidate_rate - baseline_rate, 9), is > margin
    (strictly), where a rate is failures / traces. A check in `baseline` with traces == 0 is
    ignored. A check in `baseline` that is absent from `candidate`, or has traces == 0 there,
    is missing. Checks only in `candidate` are ignored.

    Example: baseline {"a": (1, 10)}, candidate {"a": (3, 10)}, margin 0.1 -> rates 0.1 and 0.3,
    increase 0.2 > 0.1: {"passed": False, "regressions": [("a", 0.1, 0.3)], "missing": []}.

    Returns: {"passed": bool (no regression and nothing missing),
              "regressions": [(name, baseline_rate, candidate_rate)] biggest rounded rise first,
              ties by name, "missing": [names, alphabetical]}.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ci_gate() -> None:
    got = ci_gate({"a": (1, 10)}, {"a": (3, 10)}, 0.1)
    assert got == {"passed": False, "regressions": [("a", 0.1, 0.3)], "missing": []}, f"got {got!r}"
    got = ci_gate({"a": (1, 4)}, {"a": (2, 4)}, 0.25)
    assert got["passed"] and got["regressions"] == [], (
        f"got {got!r}: the rate rose by exactly the margin (0.25); a regression needs MORE than the margin")
    got = ci_gate({"a": (5, 50)}, {"a": (4, 20)}, 0.05)
    assert not got["passed"], (
        f"got {got!r}: compare rates, not counts: 5 of 50 is 0.10 and 4 of 20 is 0.20, so the check "
        "got worse although it failed fewer times")
    got = ci_gate({"a": (1, 10), "b": (2, 10)}, {"a": (1, 10)}, 0.5)
    assert not got["passed"] and got["missing"] == ["b"], (
        f"got {got!r}: a check that the candidate report no longer contains must fail the gate")
    got = ci_gate({"a": (0, 0), "b": (1, 10)}, {"b": (1, 10), "new": (9, 10)}, 0.0)
    assert got["passed"], f"got {got!r}: no baseline traces, or a check new in the candidate, is ignored"
    got = ci_gate({"a": (2, 10)}, {"a": (0, 10)}, 0.0)
    assert got["passed"] and got["missing"] == [], (
        f"got {got!r}: 0 failures on 10 traces is a perfect score, not a missing check; test the "
        "number of traces for zero, not the pair's truthiness")
    got = ci_gate({"a": (1, 10), "b": (1, 10)}, {"a": (3, 10), "b": (4, 10)}, 0.0)
    assert [r[0] for r in got["regressions"]] == ["b", "a"], (
        f"got {got!r}: list the biggest increase first")
    got = ci_gate({"a": (15, 100)}, {"a": (20, 100)}, 0.05)
    assert got["passed"], (
        f"got {got!r}: the rise is 0.05 on paper, exactly the margin, so it is not a regression; "
        "round the rise to 9 places before comparing it")
    got = ci_gate({"a": (1, 10), "b": (2, 10)}, {"a": (3, 10), "b": (4, 10)}, 0.0)
    assert [r[0] for r in got["regressions"]] == ["a", "b"], (
        f"got {got!r}: both rises are 0.2 on paper, so they tie and are listed by name; use the "
        "rounded rise for the order too")


_try("exercise 5", _check_ci_gate)

In [ ]:
def render_gate(baseline: dict, candidate: dict, result: dict, margin: float) -> None:
    """A CI-style report. It prints the exit code the job would return; it never exits, so a
    failing gate cannot end a notebook."""
    print(f"gate: a check regresses when its failure rate rises by more than {margin * 100:g} percentage points")
    print(f"{'check':<20}{'baseline':>12}{'candidate':>12}{'change':>10}   verdict")
    bad = {r[0] for r in result["regressions"]}
    for name in sorted(baseline, key=lambda n: MODES.index(n)):
        bf, bn = baseline[name]
        cf, cn = candidate.get(name, (0, 0))
        if bn == 0 or cn == 0:
            print(f"{name:<20}{'':>12}{'':>12}{'':>10}   {'MISSING' if bn and not cn else 'not rated'}")
            continue
        print(f"{name:<20}{bf / bn:>12.0%}{cf / cn:>12.0%}{(cf / cn - bf / bn) * 100:>+9.0f}p   "
              f"{'REGRESSED' if name in bad else 'ok'}")
    print("GATE:", "PASS" if result["passed"] else "FAIL")
    print("exit code:", 0 if result["passed"] else 1)


MARGIN = 0.05      # an ordinary, round choice made before looking; section 10 puts it to the test


def _show_gate() -> None:
    base, cand = run_checks(ci_base), run_checks(ci_cand)
    result = ci_gate(base, cand, MARGIN)
    render_gate(base, cand, result, MARGIN)
    before, after = failing_share(ci_base), failing_share(ci_cand)
    word = "fell" if after < before else "rose" if after > before else "did not change"
    print(f"\nthe share of traces with any failing check {word}: {before:.1%} -> {after:.1%}")
    aggregate_ok = round(after - before, 9) <= MARGIN    # the same rule as ci_gate
    print(f"a gate on that one number would {'PASS' if aggregate_ok else 'FAIL'} the candidate; "
          f"the per-check gate {'PASSES' if result['passed'] else 'FAILS'} it")
    if aggregate_ok and not result["passed"]:
        names = ", ".join(r[0] for r in result["regressions"])
        print(f"the average improved or held while {names} got worse: an average over categories "
              "can hide a regression")


_try("gate demo", _show_gate, needs=("exercise 3", "exercise 5"))

## 10. How much of that is noise?

The agent is not deterministic: the same task can go differently on another run. A gate must
not fail on that. The test is an **A/A run**: score the *baseline* agent twice, on the same
tasks, and see how often a gate at a given margin calls it a regression. Then run the
candidate several times and see how often the same margin catches what really changed. The
next cell does both, for several margins, over the runs named in `RUNS`.

In [ ]:
RUNS = 30
MARGINS = (0.0, 0.05, 0.10, 0.15, 0.20)
_CACHE: dict[tuple[str, str], dict] = {}


def _report(config_name: str, run: str) -> dict:
    if (config_name, run) not in _CACHE:
        traces, _ = generate_traces(CI_SEED, N_TRACES, {"base": BASELINE, "cand": CANDIDATE}[config_name], run=run)
        _CACHE[config_name, run] = run_checks(traces)
    return _CACHE[config_name, run]


def _show_noise() -> None:
    reference = _report("base", BASE_RUN)
    print(f"{RUNS} A/A runs (baseline against itself) and {RUNS} candidate runs, same {N_TRACES} tasks")
    print(f"{'margin':>8}  {'A/A false alarms':>17}  {'candidate caught':>17}  {'caught on unsafe_refund':>24}")
    quiet = None
    for margin in MARGINS:
        alarms, caught, caught_right, culprits = 0, 0, 0, Counter()
        for k in range(RUNS):
            aa = ci_gate(reference, _report("base", f"aa{k}"), margin)
            alarms += not aa["passed"]
            culprits.update(r[0] for r in aa["regressions"])
            cand = ci_gate(reference, _report("cand", f"cand{k}"), margin)
            caught += not cand["passed"]
            caught_right += any(r[0] == "unsafe_refund" for r in cand["regressions"])
        print(f"{margin:>8.2f}  {f'{alarms} of {RUNS}':>17}  {f'{caught} of {RUNS}':>17}  "
              f"{f'{caught_right} of {RUNS}':>24}   noisy checks: {dict(sorted(culprits.items())) or '-'}")
        if quiet is None and alarms == 0:
            quiet = (margin, caught_right)
    if quiet:
        print(f"the smallest margin tried with no false alarm: {quiet[0]:.2f}; at it the real "
              f"regression was caught in {quiet[1]} of {RUNS} candidate runs")
    else:
        print("every margin tried raised a false alarm on some A/A run")
    sizes = run_checks(ci_base)
    noisy, alarmed = Counter(), 0
    for k in range(RUNS):
        aa = ci_gate(reference, _report("base", f"aa{k}"), MARGIN)
        noisy.update(r[0] for r in aa["regressions"])
        alarmed += not aa["passed"]
    print(f"the margin used in section 9, {MARGIN:.2f}, raised a false alarm in {alarmed} of {RUNS} A/A runs, "
          "by check:", dict(sorted(noisy.items())) or "none")
    print("traces behind each check's rate:", {k: n for k, (_, n) in sizes.items()})
    if alarmed:
        print(f"so the margin chosen before looking would have failed an agent that had not changed in "
              f"{alarmed} of {RUNS} runs: it was a guess, and this table is where a guess gets tested")
    else:
        print(f"no A/A run failed at the margin chosen before looking, in {RUNS} runs")


_try("noise", _show_noise, needs=("exercise 3", "exercise 5"))

Read the table, not a rule of thumb. A margin of zero turns noise into alerts and a gate that
alerts on noise gets muted; a margin wider than the regression you care about fires only when
noise happens to push it over. Every A/A row compares with one stored baseline run, as a CI job does, and that
run is itself one draw: a baseline that came out low makes false alarms more frequent. The
margin in section 9 was a round number chosen before looking, and the A/A rows are how a margin
gets chosen from evidence instead. The
last lines name the checks behind the false alarms and the number of traces behind each check's
rate: a check rated on few traces swings more, and that is where a CI set should grow. The FAQ's
advice is that a repeatable eval set collects the important workflows and the failures you have
confirmed, and is run whenever the application changes.

What to take away. Open coding found the categories; kappa told you how much to trust one
reader; each category became a binary check you could test; and the gate compares checks, not
an average. The step after this one is a judge for what no rule can read, validated the way
you validated the checks here: against labelled examples, with its precision and recall.

## 11. Common mistakes

- **Counting phrases, not traces.** A trace described twice is one failing trace. Exercise 1
  counts each category once per trace.
- **Trusting raw agreement.** When most traces pass, two coders agree by default. Kappa
  removes that; section 5 prints both.
- **Averaging categories into one number.** An aggregate pass rate can improve while one
  category gets worse. Section 9 prints both gates on the same data.
- **Shipping a check you have not scored.** A check with no precision and recall is an
  opinion. A check that flags every retry teaches people to ignore it.
- **Treating no flags as a perfect check.** Precision is undefined when nothing was flagged;
  exercise 4 returns `None`, not 1.0.
- **A margin of zero.** Section 10 measures the false alarms it produces on an agent that did
  not change.
- **Reading precision 1.00 as proof.** A check scored against labels made by the same rule
  agrees with itself. Section 7 says which labels here are made that way.
- **Letting a check disappear.** Dropping the check that fails is the easiest way to pass; the
  gate in exercise 5 treats a missing check as a failure.

In [ ]:
# The second mistake in the list, made visible: two coders who say "pass" for almost everything.
_a = ["pass"] * 95 + ["x"] * 5
_b = ["pass"] * 95 + ["y"] * 5
_what = (f"coders who disagree on every failing trace still agree on {sum(p == q for p, q in zip(_a, _b)) / len(_a):.0%} "
         "of traces; their kappa is ")


def _kappa_of_the_coders() -> None:
    print(_what + f"{cohens_kappa(_a, _b):.3f}")


if _STATUS.get("exercise 2") != "passed":
    print(_what + "not available until exercise 2 passes")
else:
    _try("kappa of the two coders", _kappa_of_the_coders, needs=("exercise 2",))   # new input: your code runs under the limit

## 12. Self-check

Answers come with this lesson's worked solution when you enrol on Synapsa.

1. Two coders label the same traces and agree on a very high share of them, but kappa is low.
   What explains that?
   - (a) Kappa is computed on a random sample of the traces, so it is always lower than the
     share of traces on which the coders agree
   - (b) Most traces carry one label, so two coders would agree on many of them by chance,
     and kappa discounts that agreement
   - (c) Kappa penalises a coder who uses more labels than the other, whatever the labels are
   - (d) The coders must have used different spellings of the same label

2. A candidate agent lowers the share of traces on which any check fires, yet one check's
   failure rate rose clearly. Which decision rule is right?
   - (a) Ship it: the share of failing traces is the number that users experience
   - (b) Ship it, and widen the margin until the gate agrees with the aggregate
   - (c) Do not ship it, because a change that raises any rate must be rejected even if every
     rise is within noise
   - (d) Treat the rise as a regression to explain: other categories' gains can pay for it in
     the average, and the categories do not cost the same

3. `check_quality` returns `None` for precision on a check that flagged nothing. Why not 1.0?
   - (a) Precision divides by the number of flags; with no flags there is nothing it has been
     tested on, and 1.0 would claim a record it has not earned
   - (b) Because 1.0 is reserved for checks whose recall is also 1.0
   - (c) Because a check that flags nothing has been proven wrong
   - (d) Because None sorts below every number in a table

4. An unchanged agent is scored twice on the same tasks, and a gate with margin zero fails some
   of those pairs. What does that tell you?
   - (a) The checks are broken and should be rewritten
   - (b) The tasks are too easy, so the CI set should be made harder
   - (c) The gate is reading run-to-run noise as a change; choose a margin from observed A/A
     variation and from how many traces each check rates
   - (d) The candidate must be worse, since only a real change can fail a gate

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

`axial_codes` and `taxonomy_table` turned free-text notes into a counted failure taxonomy.
`cohens_kappa` measured how far two readers agree beyond chance, and the confusion table showed
which disagreements a definition fixes and which need a check. Three executable checks and
`check_quality` turned categories into binary tests with measured precision and recall, and
`ci_gate` decided a candidate change per check rather than on an average, with section 10
showing how much run-to-run noise a margin has to absorb. The next step is a judge for
the failures no rule can read, validated the way these checks were.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", lambda: (_check_axial_codes(), _check_taxonomy_table()), ()),
        ("exercise 2", _check_cohens_kappa, ()),
        ("exercise 3", _check_exercise_3, ()),
        ("exercise 4", _check_check_quality, ()),
        ("exercise 5", _check_ci_gate, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.